# 02 - Model comparison (2 baselines + 4 deep)

Compares every model trained against the 50 km-grid cache (`cache/goes_grid50_2019_2026`).
**Pure GOES/GLM signatures on the 50 km grid - no image, no climatology.** Split: train
2019-2024; 2025 by month parity - even months -> val, odd months -> test; 2026 dropped.

- **baselines** (per-cell 168-feature vectors): logistic regression, XGBoost
- **deep** (CNN / attention over the 59x95 feature grid): small 3D-ResNet, 3D-CNN,
  CNN+temporal-attention, ConvGRU

Each trainer writes `outputs/<name>.pt` (deep) / `.pkl` (tabular) + `<name>_results.npz`
(deep epoch curves). This notebook reloads each, runs inference on val + test, and reports
**AUPRC (+lift), P/R/F1/CSI** (threshold from val applied to test), training curves, and
prediction maps. The **base rate** is the reference floor.

Train first (from `notebooks/model/`):

```
NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=2 trainers/cnn3d.py   # + resnet3d, cnn_attn, convgru
python trainers/logreg.py    # tabular: plain python, NOT torchrun
python trainers/xgb.py
```

## 0. Setup, registry, and artifact availability

In [ ]:
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
OUT_DIR = MODEL_DIR / "outputs"
sys.path.insert(0, str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR / "trainers"))

from config import STATES_GEOJSON, build_grid_cells
from gridindex import build_pix2cell
import cnn3d, resnet3d, cnn_attn, convgru   # noqa: E401  (deep trainers)
import logreg, xgb                                      # noqa: E401  (tabular trainers)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# shared grid / land mask / splits (the same filtered split every trainer uses)
p2c, GRID_R, GRID_C, land = build_pix2cell()
tr_days, va_days, te_days = cnn3d.load_splits()
CACHE_DIR = cnn3d.CACHE_DIR


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


BASE = {"val": base_rate(va_days), "test": base_rate(te_days)}

# grid polygons + state outlines (Albers) for the maps
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)

# 6-model registry: deep (reload .pt into FloodNet) + tabular (reload .pkl)
MODELS = {
    "3D-ResNet": dict(kind="deep", mod=resnet3d, ckpt="resnet3d.pt",
                      res="resnet3d_results.npz", c="#d62828", cmap="Reds"),
    "3D-CNN":    dict(kind="deep", mod=cnn3d, ckpt="cnn3d.pt",
                      res="cnn3d_results.npz", c="#e09f3e", cmap="Oranges"),
    "CNN+attn":  dict(kind="deep", mod=cnn_attn, ckpt="cnn_attn.pt",
                      res="cnn_attn_results.npz", c="#2a9d8f", cmap="GnBu"),
    "ConvGRU":   dict(kind="deep", mod=convgru, ckpt="convgru.pt",
                      res="convgru_results.npz", c="#6a4c93", cmap="Purples"),
    "LogReg":    dict(kind="tab", mod=logreg, pkl="logreg.pkl",
                      c="#8d99ae", cmap="bone_r"),
    "XGBoost":   dict(kind="tab", mod=xgb, pkl="xgb.pkl",
                      c="#386641", cmap="YlGn"),
}


def artifact(mi):
    return OUT_DIR / (mi["ckpt"] if mi["kind"] == "deep" else mi["pkl"])


AVAIL = [n for n, mi in MODELS.items() if artifact(mi).exists()]

print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells")
print(f"days: train {len(tr_days)}  val {len(va_days)}  test {len(te_days)}")
print(f"base rate: val {BASE['val']:.4f}  test {BASE['test']:.4f}")
for n, mi in MODELS.items():
    print(f"  {n:10s} {mi['kind']:4s} -> {'TRAINED' if n in AVAIL else 'missing (train it)'}")

## 1. Training curves (deep models)

Train loss and **train/val** PR-AUC across epochs from each `*_results.npz` (early-stopped on
val). Test is intentionally **not** tracked per epoch (only at the end, in the metrics table)
to avoid biasing model selection. Dotted line = val base rate.

In [ ]:
deep_avail = [n for n in AVAIL if MODELS[n]["kind"] == "deep"]
PR = [("train", "#1d6fb8", "-o", 3), ("val", "#e09f3e", "-s", 4)]

for name in deep_avail:
    path = OUT_DIR / MODELS[name]["res"]
    if not path.exists():
        print(f"{name}: no results npz")
        continue
    h = np.load(path)["hist"]          # epoch, lr, loss, tr_pr, va_pr
    if h.ndim != 2 or h.shape[1] != 5 or len(h) == 0:
        print(f"{name}: empty/bad hist")
        continue
    ep = h[:, 0]
    fig, (a_loss, a_pr) = plt.subplots(1, 2, figsize=(13, 4.4))
    a_loss.plot(ep, h[:, 2], "-o", color="#1d6fb8", ms=4, label="train loss")
    for split, col, mk, k in PR:
        a_pr.plot(ep, h[:, k], mk, color=col, ms=4, label=split)
    a_pr.axhline(BASE["val"], ls=":", color="black", lw=1.4,
                 label=f"val base ({BASE['val']:.4f})")
    a_loss.set(xlabel="epoch", ylabel="combined loss", title=f"{name} - train loss")
    a_pr.set(xlabel="epoch", ylabel="PR-AUC (land)", title=f"{name} - PR-AUC (train/val)")
    for ax in (a_loss, a_pr):
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    b = int(h[:, 4].argmax())
    print(f"{name}: best val PR-AUC {h[b,4]:.4f} @ epoch {int(h[b,0])}")

## 2. Inference on val + test

Reload each trained model and predict per-cell probabilities `(N, 59, 95)`. Deep models
reload their `.pt` into `FloodNet`; tabular models reload their `.pkl` and use
`predict_grids`. Cached once for the metrics table and maps below.

In [ ]:
@torch.no_grad()
def infer(name, days):
    """Reload model `name` and predict -> (probs, trues), each (len(days), 59, 95)."""
    mi = MODELS[name]
    if mi["kind"] == "deep":
        mod = mi["mod"]
        net = mod.FloodNet().to(DEVICE)
        net.load_state_dict(torch.load(OUT_DIR / mi["ckpt"], map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=16, num_workers=8)
        for seq, summ, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                p = torch.sigmoid(net(seq.to(DEVICE).float(), summ.to(DEVICE).float(),
                                      t.to(DEVICE).float())).squeeze(1)
            probs.append(p.float().cpu().numpy())
            trues.append(y.numpy())
        del net
        torch.cuda.empty_cache()
        return np.concatenate(probs), np.concatenate(trues)
    # tabular
    payload = pickle.load(open(OUT_DIR / mi["pkl"], "rb"))
    return mi["mod"].predict_grids(payload, days, land)


val_pred, test_pred = {}, {}
for name in AVAIL:
    print(f"inferring {name} ...", end=" ", flush=True)
    val_pred[name] = infer(name, va_days)
    test_pred[name] = infer(name, te_days)
    print("done")

## 3. Metrics table

AUPRC + P/R/F1/CSI (exact and 1-grid neighbourhood) on val and test. The threshold is the
best-F1 point on **val**, applied to test (the honest operating point). `xbase` = AUPRC /
base rate. 1-grid metrics credit a prediction within one cell of a true flood.

In [ ]:
import pandas as pd


def _dilate_1grid(mask):
    out = mask.copy()
    out[:-1, :] |= mask[1:, :]
    out[1:, :] |= mask[:-1, :]
    out[:, :-1] |= mask[:, 1:]
    out[:, 1:] |= mask[:, :-1]
    out[:-1, :-1] |= mask[1:, 1:]
    out[1:, 1:] |= mask[:-1, :-1]
    out[:-1, 1:] |= mask[1:, :-1]
    out[1:, :-1] |= mask[:-1, 1:]
    return out


def full_metrics(probs, trues, threshold=None):
    p = probs[:, land].ravel()
    t = trues[:, land].ravel().astype(np.int32)
    prauc = average_precision_score(t, p)
    if threshold is None:
        pr_c, rc_c, thr_c = precision_recall_curve(t, p)
        f1_c = 2 * pr_c * rc_c / (pr_c + rc_c + 1e-9)
        threshold = float(thr_c[np.argmax(f1_c[:-1])])
    yb = (p >= threshold).astype(np.int32)
    tp = int((yb * t).sum()); fp = int((yb * (1 - t)).sum()); fn = int(((1 - yb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    h_nb = m_nb = fa_nb = 0
    for i in range(len(probs)):
        yt = (trues[i] > 0.5) & land
        yp = (probs[i] >= threshold) & land
        h_nb += int((yt & (_dilate_1grid(yp) & land)).sum())
        m_nb += int((yt & ~(_dilate_1grid(yp) & land)).sum())
        fa_nb += int((yp & ~(_dilate_1grid(yt) & land)).sum())
    prec1 = h_nb / (h_nb + fa_nb + 1e-9); rec1 = h_nb / (h_nb + m_nb + 1e-9)
    f1_1 = 2 * prec1 * rec1 / (prec1 + rec1 + 1e-9); csi1 = h_nb / (h_nb + m_nb + fa_nb + 1e-9)
    return dict(prauc=prauc, thr=threshold, prec=prec, rec=rec, f1=f1, csi=csi,
                prec1=prec1, rec1=rec1, f1_1=f1_1, csi1=csi1)


rows = []
THR = {}
for name in AVAIL:
    vm = full_metrics(*val_pred[name], threshold=None)
    THR[name] = vm["thr"]
    tm = full_metrics(*test_pred[name], threshold=vm["thr"])
    for split, m in [("val", vm), ("test", tm)]:
        rows.append(dict(model=name, split=split, AUPRC=m["prauc"],
                         xbase=m["prauc"] / BASE[split], thr=m["thr"],
                         P=m["prec"], R=m["rec"], F1=m["f1"], CSI=m["csi"],
                         P1=m["prec1"], R1=m["rec1"], F1_1=m["f1_1"], CSI1=m["csi1"]))

tbl = pd.DataFrame(rows)
test_tbl = tbl[tbl.split == "test"].sort_values("AUPRC", ascending=False)
print(f"base rate: val {BASE['val']:.4f}  test {BASE['test']:.4f}\n")
print("TEST (sorted by AUPRC):")
fmt = {c: "{:.3f}".format for c in ["AUPRC", "xbase", "thr", "P", "R", "F1", "CSI",
                                    "P1", "R1", "F1_1", "CSI1"]}
display(test_tbl.set_index("model").drop(columns="split").style.format(fmt))
_b = test_tbl.iloc[0]
print(f"\n>>> best test AUPRC: {_b['model']}  {_b['AUPRC']:.4f}  ({_b['xbase']:.1f}x base)")
tbl.round(4)

## 4. Prediction maps — random test days

Grid of **rows = models** (ground truth on top) × **columns = sample test days**. Change
`SEED` to resample which days are shown; `N_SHOW` sets how many. This cell binarizes each
model's prediction at its best-val-F1 threshold; the next cell shows the raw probability
heatmap. Per-panel AUPRC is annotated below each map.

In [ ]:
assert AVAIL, "no trained models yet - train them first (see section 0)"
SEED = 99        # <- change to resample which test days are shown
N_SHOW = 5


def _draw(ax, values, cmap, norm=None, title=None, ylabel=None, sub=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    kw = dict(norm=norm) if norm is not None else dict(vmin=0, vmax=1)
    gdf.plot(column="v", cmap=cmap, ax=ax, zorder=1, edgecolor="none", **kw)
    states.boundary.plot(ax=ax, color="0.5", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:  ax.set_title(title, fontsize=11, fontweight="bold")
    if ylabel: ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")
    if sub:    ax.set_xlabel(sub, fontsize=8, color="0.3")


rng = np.random.default_rng(SEED)
sel = sorted(rng.choice(len(te_days), size=min(N_SHOW, len(te_days)), replace=False))
row_names = ["Ground truth"] + AVAIL                      # rows = GT + each model
nrow, ncol = len(row_names), len(sel)

fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 3.2 * nrow), squeeze=False)
for c, i in enumerate(sel):
    gt = test_pred[AVAIL[0]][1][i]
    yt = gt[land].ravel().astype(int)
    _draw(axes[0, c], gt, "Greens",
          title=f"{te_days[i]}  ({int((gt[land] > 0).sum())} floods)",
          ylabel="Ground truth" if c == 0 else None)
    for r, name in enumerate(AVAIL, start=1):
        pr = test_pred[name][0][i]
        ap = average_precision_score(yt, pr[land].ravel()) if yt.sum() else float("nan")
        _draw(axes[r, c], (pr >= THR[name]).astype(float), MODELS[name]["cmap"],
              ylabel=name if c == 0 else None, sub=f"AUPRC {ap:.3f}")

fig.suptitle(f"Test days (seed {SEED}) - binary predictions @ best-val-F1 threshold",
             fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()

## 5. Probability heatmaps — same days

Same rows-models × columns-days grid, but each model panel shows the **raw predicted
probability** (not thresholded), on a shared 0-to-max colour scale so models are
comparable. Reuses the `SEED`/`sel` chosen above.

In [ ]:
PCMAP = "viridis"
vmax = max(float(test_pred[n][0][sel][:, land].max()) for n in AVAIL)
vmax = max(vmax, 1e-3)
pnorm = mcolors.Normalize(0, vmax)

fig, axes = plt.subplots(nrow, ncol, figsize=(3.5 * ncol, 3.2 * nrow), squeeze=False)
for c, i in enumerate(sel):
    gt = test_pred[AVAIL[0]][1][i]
    yt = gt[land].ravel().astype(int)
    _draw(axes[0, c], gt, "Greens",
          title=f"{te_days[i]}  ({int((gt[land] > 0).sum())} floods)",
          ylabel="Ground truth" if c == 0 else None)
    for r, name in enumerate(AVAIL, start=1):
        pr = test_pred[name][0][i]
        ap = average_precision_score(yt, pr[land].ravel()) if yt.sum() else float("nan")
        _draw(axes[r, c], pr, PCMAP, norm=pnorm,
              ylabel=name if c == 0 else None, sub=f"AUPRC {ap:.3f}")

fig.subplots_adjust(right=0.9)
cax = fig.add_axes([0.92, 0.15, 0.014, 0.7])
fig.colorbar(plt.cm.ScalarMappable(cmap=PCMAP, norm=pnorm), cax=cax,
             label="predicted probability")
fig.suptitle(f"Test days (seed {SEED}) - predicted probability (shared scale 0-{vmax:.2f})",
             fontsize=14, fontweight="bold", y=1.002)
plt.show()

## 6. Spatial skill of the best model

For the top model (by test AUPRC), aggregate over all test days: **observed**
flood-days per cell vs **predicted** flood-days (binarized at its val threshold),
and the **net bias** (pred - obs). Shows *where* across CONUS the model
over- or under-predicts floods.

In [ ]:
name = test_tbl.iloc[0]["model"]                 # best by test AUPRC
probs, trues = test_pred[name]
obs = trues.sum(0).astype(float)                 # observed flood-days per cell
pred = (probs >= THR[name]).sum(0).astype(float)  # predicted flood-days per cell
bias = pred - obs
mx = max(obs.max(), pred.max(), 1.0)
b = max(abs(bias).max(), 1.0)
cnt_norm = mcolors.Normalize(0, mx)
div_norm = mcolors.TwoSlopeNorm(0, -b, b)

fig, axes = plt.subplots(1, 3, figsize=(4.2 * 3, 4.2))
_draw(axes[0], obs, "magma_r", norm=cnt_norm, title="Observed flood-days")
_draw(axes[1], pred, "magma_r", norm=cnt_norm, title=f"{name}: predicted flood-days")
_draw(axes[2], bias, "RdBu_r", norm=div_norm, title="Net bias (pred - obs)")
fig.colorbar(plt.cm.ScalarMappable(cmap="magma_r", norm=cnt_norm), ax=axes[:2],
             fraction=0.024, pad=0.01, label="flood-days over test")
fig.colorbar(plt.cm.ScalarMappable(cmap="RdBu_r", norm=div_norm), ax=axes[2],
             fraction=0.046, pad=0.01, label="pred - obs")
fig.suptitle(f"Spatial skill of best model ({name}) over {len(te_days)} test days",
             fontsize=13, fontweight="bold")
plt.show()